In [ ]:
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd

# Pré-processamento
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

# Seleção de modelo
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.model_selection import GridSearchCV, ParameterGrid 

# Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


from sklearn.svm import SVC

# Metricas
from sklearn.metrics import  confusion_matrix

In [ ]:
# Bases de dados
DF_VEICULOS = "Veiculos - Dados.csv"
DF_DIABETES = "Diabetes - Dados.csv"


# Seed Global
RAND_SEED = 202613 # Ano atual + número do Grupo
np.random.seed(RAND_SEED)

# Tamanho da base de teste: 30%
TEST_SIZE = 0.3

# Largura máxima das colunas a serem exibidas
pd.set_option('display.max_colwidth', None)

In [ ]:
# Carrega DataFrame
def load_df(df_name, column_to_drop=""):
    """Função para carregar o csv e se informado, remover colunas"""
    try:
        df = pd.read_csv(df_name, sep=',')
        if column_to_drop:
            df = df.drop(column_to_drop, axis=1)
        return df
    
    except:
        print("Erro ao processar o arquivo", df_name)

In [ ]:
# Função para executar o experimento

from sklearn.metrics import accuracy_score

def executar_experimento(config, X, y, classes, test_size=TEST_SIZE):

    modelo = config["modelo"]
    parametros = config["parametros"]
    
    # ======================================================
    # HOLD-OUT
    # ======================================================
    if config["avaliacao"] == "hold_out":

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=test_size,
            stratify=y,
            random_state=RAND_SEED
        )

        melhor_score = -1
        melhor_param = None
        melhor_pipeline = None
        
        for p in ParameterGrid(parametros):
            modelo.set_params(**p)
            pipeline = Pipeline([("scaler", MinMaxScaler()), ("modelo", modelo)])
            pipeline.fit(X_train, y_train)

            y_pred = pipeline.predict(X_test)
            score = accuracy_score(y_test, y_pred)

            if score > melhor_score:
                melhor_score = score
                melhor_param = p
                melhor_pipeline = pipeline
                melhor_y_pred = y_pred

        return {
            "melhor_parametro": melhor_param,
            "score": round(melhor_score, 2),
            "modelo": melhor_pipeline,
            "confusion_matrix": confusion_matrix(y_test, melhor_y_pred, labels=classes) # calcula a matriz de confusão
        }

    # ======================================================
    # CROSS-VALIDATION
    # ======================================================
    elif config["avaliacao"] == "cross_validation":
        #Ajuste do nome dos parametros devido a passagem pelo pipeline
        param_grid = {
                    f'modelo__{k}': v
                    for k, v in config["parametros"].items()
                }
        
        pipeline = Pipeline([("scaler", MinMaxScaler()), ("modelo", modelo)])
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=config.get("cv", 5),
            scoring=config["metrica"],
            n_jobs=-1
        )
        
        grid.fit(X, y)
        y_pred = cross_val_predict(pipeline, X, y, cv=config.get("cv", 5)) # Predições fora da amostra

        return {
            "melhor_parametro": grid.best_params_,
            "score": round(grid.best_score_, 2),
            "modelo": grid.best_estimator_,
            "confusion_matrix": confusion_matrix(y, y_pred, labels=classes) # calcula a matriz de confusão
        }

    else:
        raise ValueError(
            f"Tipo de avaliação inválido: {config['avaliacao']}"
        )


In [ ]:
# Função que treina os modelos para uma base de dados (X e y) e retorna o resultado do experimento

def testa_modelos(modelos, X, y, classes):
    # Dicionário para armazenar os resultados
    resultados = {}

    # Percorre todos os modelos configurados
    for chave, config in modelos.items():

        print(f"Executando: {config['nome']}")

        try:
            resultado = executar_experimento(config, X, y, classes)

            # Guarda o resultado completo
            resultados[chave] = {
                "nome": config["nome"],
                "avaliacao": config["avaliacao"],
                "metrica": config["metrica"],
                "melhor_parametro": resultado["melhor_parametro"],
                "score": resultado["score"],
                "modelo": resultado["modelo"],
                "confusion_matrix": resultado["confusion_matrix"]
            }

            print(f"  Melhor score: {resultado['score']:.2f}")
            print(f"  Melhor parâmetro: {resultado['melhor_parametro']}")

        except Exception as e:
            print(f"  Erro ao executar {config['nome']}: {e}")

        print("-" * 80)
    return resultados

In [ ]:
modelos = {
    "KNN": {
        "nome": "KNN",
        "modelo": KNeighborsClassifier(),
        "avaliacao": "hold_out",
        "parametros": {
                    'n_neighbors': [1, 3, 5, 7, 9],
        },
        "metrica": "accuracy"
    },
    
    "RNA_HO": {
        "nome": "RNA - Hold Out",
        "modelo": MLPClassifier(
            max_iter=2000,
            random_state=RAND_SEED
        ),
        "avaliacao": "hold_out",
        "parametros": {
            'hidden_layer_sizes': [(10,), (20,), (30, 10)],
            'learning_rate_init': [0.001, 0.01, 0.1]
        },
        "metrica": "accuracy"
    },
    
    "RNA_CV": {
        "nome": "RNA - Cross Validation",
        "modelo": MLPClassifier(
            max_iter=2000,
            random_state=RAND_SEED
        ),
        "avaliacao": "cross_validation",
        "parametros": {
            "hidden_layer_sizes": [(10,), (20,), (30, 10)],
            "learning_rate_init": [0.001, 0.01, 0.1]
        },
        "metrica": "accuracy"
    },

    "SVM_HO": {
        "nome": "SVM - Hold Out",
        "modelo": SVC(random_state=RAND_SEED),
        "avaliacao": "hold_out",
        "parametros": {
            'C': [1, 10, 50, 100],
            'gamma': ['auto', 'scale']
        },
        "metrica": "accuracy"
    },
    
    "SVM_CV": {
        "nome": "SVM - Cross Validation",
        "modelo": SVC(random_state=RAND_SEED),
        "avaliacao": "cross_validation",
        "parametros": {
            'C': [1, 10, 50, 100],
            'gamma': ['auto', 'scale']
        },
        "metrica": "accuracy"
    },
    
    "RF_HO": {
        "nome": "Random Forest - Hold Out",
        "modelo": RandomForestClassifier(
            random_state=RAND_SEED
        ),
        "avaliacao": "hold_out",
        "parametros": {
            "n_estimators": [10, 50, 100, 200],
            "max_depth": [None, 1, 10, 20]
        },
        "metrica": "accuracy"
    },
    
    "RF_CV": {
        "nome": "Random Forest - Cross Validation",
        "modelo": RandomForestClassifier(
            random_state=RAND_SEED
        ),
        "avaliacao": "cross_validation",
        "cv": 5,
        "parametros": {
            "n_estimators": [10, 50, 100, 200],
            "max_depth": [None, 1, 10, 20]
        },
        "metrica": "accuracy"
    }
}

In [ ]:
df = load_df(DF_VEICULOS, 'a')

y = df['tipo']
X = df.drop('tipo', axis = 1)

columns = list(X.columns)
classes = y.unique().tolist()

# # Restaura os nomes das colunas
X = pd.DataFrame(X, columns=columns)

In [ ]:
resultados_veiculos = testa_modelos(modelos, X, y, classes)